# 🔥 FIXED: Construction Cost Prediction with Country Residual Modeling

## Critical Fixes Applied:
1. **Country Residual Modeling** - Train on deviations from country mean
2. **Drop GDP** - Remove country identifier causing shortcut learning
3. **Enhanced Satellite Features** - NDVI, NDBI, texture, percentiles
4. **GroupKFold by Location** - Prevent leakage

In [1]:
!pip install -q rasterio lightgbm scikit-learn pandas numpy scipy tqdm


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from tqdm.auto import tqdm
import rasterio
from scipy.ndimage import generic_filter

c:\Users\Prawin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
BASE = Path('d:/Projects/Construction-cost')
DATA_DIR = BASE / 'datasets'
train = pd.read_csv(DATA_DIR / 'train_tabular.csv')
eval_df = pd.read_csv(DATA_DIR / 'evaluation_tabular_no_target.csv')
print(f'Train: {train.shape}, Eval: {eval_df.shape}')

Train: (1024, 23), Eval: (1024, 22)


In [4]:
# Fill missing values
train['tropical_cyclone_wind_risk'] = train['tropical_cyclone_wind_risk'].fillna('Low')
eval_df['tropical_cyclone_wind_risk'] = eval_df['tropical_cyclone_wind_risk'].fillna('Low')

# Encode booleans
bool_cols = ['developed_country', 'landlocked', 'access_to_airport', 'access_to_port', 
             'access_to_highway', 'access_to_railway', 'flood_risk_class']
for col in bool_cols:
    train[col] = (train[col] == 'Yes').astype(int)
    eval_df[col] = (eval_df[col] == 'Yes').astype(int)

# Ordinal encode risks
risk_map = {'Very Low': 0, 'Low': 1, 'Moderate': 2, 'High': 3, 'Very High': 4}
for col in ['seismic_hazard_zone', 'tropical_cyclone_wind_risk', 'tornadoes_wind_risk']:
    train[col] = train[col].map(risk_map)
    eval_df[col] = eval_df[col].map(risk_map)

# Label encode categoricals
for col in ['country', 'region_economic_classification', 'koppen_climate_zone']:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    eval_df[col] = le.transform(eval_df[col])

## 🔥 Enhanced Satellite Feature Extraction

In [ ]:
def extract_satellite_features(tif_path):
    """Extract rich satellite features for within-country variation"""
    try:
        with rasterio.open(tif_path) as src:
            bands = src.read()
            
            if bands.shape[0] >= 4:
                blue, green, red, nir = bands[0], bands[1], bands[2], bands[3]
                
                # NDVI
                ndvi = np.where((nir + red) > 0, (nir - red) / (nir + red), 0)
                
                # NDBI (built-up proxy)
                ndbi = np.where((red + nir) > 0, (red - nir) / (red + nir), 0)
                
                # Texture
                red_var = generic_filter(red.astype(float), np.var, size=5)
                
                return {
                    'ndvi_mean': np.nanmean(ndvi),
                    'ndvi_std': np.nanstd(ndvi),
                    'ndvi_p10': np.nanpercentile(ndvi, 10),
                    'ndvi_p90': np.nanpercentile(ndvi, 90),
                    'ndbi_mean': np.nanmean(ndbi),
                    'ndbi_std': np.nanstd(ndbi),
                    'red_mean': np.nanmean(red),
                    'red_std': np.nanstd(red),
                    'red_texture': np.nanmean(red_var),
                    'nir_mean': np.nanmean(nir),
                    'blue_mean': np.nanmean(blue),
                    'green_mean': np.nanmean(green),
                }
            else:
                band = bands[0]
                return {
                    'brightness_mean': np.nanmean(band),
                    'brightness_std': np.nanstd(band),
                    'brightness_p10': np.nanpercentile(band, 10),
                    'brightness_p90': np.nanpercentile(band, 90),
                }
    except:
        return {}

print('Extracting satellite features for train...')
sentinel_feats = []
viirs_feats = []
for idx, row in tqdm(train.iterrows(), total=len(train)):
    s_path = DATA_DIR / 'train_composite' / row['sentinel2_tiff_file_name']
    v_path = DATA_DIR / 'train_composite' / row['viirs_tiff_file_name']
    sentinel_feats.append(extract_satellite_features(s_path))
    viirs_feats.append(extract_satellite_features(v_path))

train_sentinel = pd.DataFrame(sentinel_feats).add_prefix('sentinel_')
train_viirs = pd.DataFrame(viirs_feats).add_prefix('viirs_')
train = pd.concat([train.reset_index(drop=True), train_sentinel, train_viirs], axis=1)

print('Extracting satellite features for eval...')
sentinel_feats = []
viirs_feats = []
for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    s_path = DATA_DIR / 'evaluation_composite' / row['sentinel2_tiff_file_name']
    v_path = DATA_DIR / 'evaluation_composite' / row['viirs_tiff_file_name']
    sentinel_feats.append(extract_satellite_features(s_path))
    viirs_feats.append(extract_satellite_features(v_path))

eval_sentinel = pd.DataFrame(sentinel_feats).add_prefix('sentinel_')
eval_viirs = pd.DataFrame(viirs_feats).add_prefix('viirs_')
eval_df = pd.concat([eval_df.reset_index(drop=True), eval_sentinel, eval_viirs], axis=1)

print(f'Train: {train.shape}, Eval: {eval_df.shape}')

Extracting satellite features for train...


  5%|▌         | 52/1024 [16:10<4:39:14, 17.24s/it]

## 🔥 Country Residual Modeling

In [ ]:
# Compute country means
country_means = train.groupby('country')['construction_cost_per_m2_usd'].mean()
print('Country means:')
print(country_means)

# Create residual target
train['country_mean'] = train['country'].map(country_means)
train['target_residual'] = train['construction_cost_per_m2_usd'] - train['country_mean']

print(f"\nOriginal - Mean: {train['construction_cost_per_m2_usd'].mean():.1f}, Std: {train['construction_cost_per_m2_usd'].std():.1f}")
print(f"Residual - Mean: {train['target_residual'].mean():.1f}, Std: {train['target_residual'].std():.1f}")

eval_df['country_mean'] = eval_df['country'].map(country_means)

## 🔥 Model Training - DROP GDP, Use GroupKFold

In [ ]:
# Define features - DROP GDP!
drop_cols = ['data_id', 'geolocation_name', 'quarter_label', 'sentinel2_tiff_file_name', 
             'viirs_tiff_file_name', 'construction_cost_per_m2_usd', 'target_residual',
             'country_mean', 'deflated_gdp_usd']  # DROP GDP!

feature_cols = [c for c in train.columns if c not in drop_cols]
print(f'Using {len(feature_cols)} features')

X = train[feature_cols].fillna(0)
y_residual = train['target_residual']
groups = train['geolocation_name']

# GroupKFold validation
gkf = GroupKFold(n_splits=5)
cv_scores = []
models = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y_residual, groups)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y_residual.iloc[train_idx], y_residual.iloc[val_idx]
    
    model = lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42 + fold,
        verbose=-1
    )
    
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
              callbacks=[lgb.early_stopping(50, verbose=False)])
    
    # Predict residuals and add back country mean
    pred_residual = model.predict(X_val)
    pred_final = pred_residual + train.iloc[val_idx]['country_mean'].values
    true_final = train.iloc[val_idx]['construction_cost_per_m2_usd'].values
    
    mae = mean_absolute_error(true_final, pred_final)
    cv_scores.append(mae)
    models.append(model)
    
    print(f'Fold {fold+1} MAE: {mae:.2f}')

print(f'\nMean CV MAE: {np.mean(cv_scores):.2f} ± {np.std(cv_scores):.2f}')

## Generate Predictions

In [ ]:
# Make predictions on eval set
X_eval = eval_df[feature_cols].fillna(0)

# Average predictions from all folds
residual_preds = np.mean([m.predict(X_eval) for m in models], axis=0)

# Add back country mean
final_preds = residual_preds + eval_df['country_mean'].values

# Check prediction distribution
print('\nPrediction statistics:')
print(f'Mean: {final_preds.mean():.1f}')
print(f'Std: {final_preds.std():.1f}')
print(f'Min: {final_preds.min():.1f}')
print(f'25%: {np.percentile(final_preds, 25):.1f}')
print(f'50%: {np.percentile(final_preds, 50):.1f}')
print(f'75%: {np.percentile(final_preds, 75):.1f}')
print(f'Max: {final_preds.max():.1f}')

# Create submission
submission = pd.DataFrame({
    'data_id': eval_df['data_id'],
    'construction_cost_per_m2_usd': final_preds
})

submission.to_csv(BASE / 'submission_residual_model.csv', index=False)
print(f'\nSubmission saved to {BASE / "submission_residual_model.csv"}')
print(f'Shape: {submission.shape}')
submission.head(10)